# Chapter 8 · Quantum Neural Network (QNN)

## Objectives

1. Understand the architecture of a quantum neural network (QNN) based on variational circuits.
2. Implement a two-class quantum classifier using `SamplerQNN` from Qiskit.
3. Train the network with gradient based on `parameter-shift rule`.
4. Compare the performance with an equivalent classical neural network.

---

## 8B.1 QNN Architecture

A QNN combines:

- **Embedding layer**: Circuit $U(\mathbf{x})$ that encodes the classical data into the quantum state.
- **Variational layer**: Circuit $W(\boldsymbol{\theta})$ with trainable parameters.
- **Measurement**: The expected value of an observable serves as the network output.

The parameter-shift rule allows computing exact gradients:

$$\frac{\partial E}{\partial \theta_i} = \frac{E(\theta_i + \pi/2) - E(\theta_i - \pi/2)}{2}$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import COBYLA, SPSA

print('QNN modules loaded.')

In [ ]:
# ── Dataset: concentric circles ───────────────────────────────────
np.random.seed(0)
X, y = make_circles(n_samples=80, noise=0.1, factor=0.5)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

# Map labels to {-1, +1} for the QNN
y_train_qnn = 2 * y_train - 1
y_test_qnn  = 2 * y_test - 1

print(f'Dataset: {len(X_train)} train, {len(X_test)} test')

# Visualization
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y==0, 0], X[y==0, 1], c='#58a6ff', label='Class 0')
ax.scatter(X[y==1, 0], X[y==1, 1], c='#f78166', label='Class 1')
ax.set_title('Dataset: concentric circles')
ax.legend()
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
# ── QNN Circuit ───────────────────────────────────────────────────
n_qubits = 2

feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=1)
ansatz      = RealAmplitudes(n_qubits, reps=2, entanglement='linear')

# Full circuit: embedding + variational
qnn_circuit = QuantumCircuit(n_qubits)
qnn_circuit.compose(feature_map, inplace=True)
qnn_circuit.compose(ansatz,      inplace=True)

print('QNN Circuit:')
print(qnn_circuit.decompose().draw('text'))

print(f'\nInput parameters (data): {feature_map.num_parameters}')
print(f'Trainable parameters:    {ansatz.num_parameters}')

In [ ]:
# ── Define and train the QNN ──────────────────────────────────────
from qiskit.quantum_info import SparsePauliOp

observable = SparsePauliOp.from_list([('ZI', 1.0)])

estimator_qnn = EstimatorQNN(
    circuit=qnn_circuit,
    observables=observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

# Quantum classifier
loss_history = []

def callback(weights, obj_val):
    loss_history.append(obj_val)

qnn_classifier = NeuralNetworkClassifier(
    neural_network=estimator_qnn,
    optimizer=COBYLA(maxiter=150),
    callback=callback,
)

print('Training the QNN...')
qnn_classifier.fit(X_train, y_train_qnn)
print('Training completed.')

# Evaluation
y_pred = qnn_classifier.predict(X_test)
acc = accuracy_score(y_test_qnn, y_pred)
print(f'\nQNN Accuracy: {acc:.4f}')

In [ ]:
# Learning curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color='#58a6ff', linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Learning curve — QNN classifier')
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 8B.2 Proposed Exercises

1. Replace the `COBYLA` optimizer with `ADAM` and manually implement the parameter-shift rule to estimate the gradients.

2. Increase the number of ansatz layers. How do accuracy and training time evolve?

3. Investigate the **barren plateau** problem in deep QNNs: why does the gradient tend to zero as the number of qubits and layers grows?